In [27]:
# Importing all the necessary packages. 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from collections import defaultdict, deque



In [ ]:
# Importing the pre-defined functions from the milestone notebook. 
from ipynb.fs.defs.Project_Milestone_clean import compute_elo, compute_h2h, compute_past_game_counts, compute_last_n_form, compute_last_5_goals_scored



In [29]:
# Load in the main dataset - the historical matches dataset

df_match_history = pd.read_csv("./../data/results.csv")
df_match_history.shape

(49287, 9)

In [ ]:
# generate the features just like how we did in the milestone notebook. 
df_matches = compute_elo(df_match_history)
df_matches = compute_h2h(df_matches)
df_matches = compute_past_game_counts(df_matches)
df_matches = compute_last_n_form(df_matches)

In [ ]:
# generate the features just like how we did in the milestone notebook. 

# Normalize the weighted score of the home and away team with the number of matches played in the past. 
df_matches["normalized_points_home_team"] = np.where(df_matches["home_count_last5"]==0, 0, df_matches["last_5_points_home_team"]/df_matches["home_count_last5"])
df_matches["normalized_points_away_team"] = np.where(df_matches["away_count_last5"]==0, 0, df_matches["last_5_points_away_team"]/df_matches["away_count_last5"])

# Generate the score difference and sum to use as our features
df_matches["pts_pg_diff_last5"] = df_matches["normalized_points_home_team"] - df_matches["normalized_points_away_team"]
df_matches["pts_pg_sum_last5"] = df_matches["normalized_points_home_team"] + df_matches["normalized_points_away_team"]

In [ ]:
# generate the features just like how we did in the milestone notebook. 

df_matches = compute_last_5_goals_scored(df_matches)

In [ ]:
# generate the features just like how we did in the milestone notebook. 

# Normalize the number of goals scored and conceded for each teams based on the number of matches they've played. 
df_matches["normalized_goals_for_home_team"] = np.where(df_matches["home_count_last5"]==0, 0, df_matches["home_num_goals_scored_last_5_games"]/df_matches["home_count_last5"])
df_matches["normalized_goals_for_away_team"] = np.where(df_matches["away_count_last5"]==0, 0, df_matches["away_num_goals_scored_last_5_games"]/df_matches["away_count_last5"])
df_matches["normalized_goals_against_home_team"] = np.where(df_matches["home_count_last5"]==0, 0, df_matches["home_num_goals_conceded_last_5_games"]/df_matches["home_count_last5"])
df_matches["normalized_goals_against_away_team"] = np.where(df_matches["away_count_last5"]==0, 0, df_matches["away_num_goals_conceded_last_5_games"]/df_matches["away_count_last5"])

# Generate the diff columns to use as our features. 
df_matches["gf_pg_diff_last5"] = df_matches["normalized_goals_for_home_team"] - df_matches["normalized_goals_for_away_team"]
df_matches["ga_pg_diff_last5"] = df_matches["normalized_goals_against_home_team"] - df_matches["normalized_goals_against_away_team"]

In [ ]:
# Get the world cup group stages matches with the features we've generated. 
wc_matches = df_matches[(df_matches["tournament"] == "FIFA World Cup") & (df_matches["date"] >= "2026-06-11")]

In [ ]:
# get all the countries that are in the world cup matches. 
countries_in_wc = set(wc_matches["home_team"].unique()).union(set(wc_matches["away_team"].unique()))

In [ ]:
# get the latest FIFA ranking
df_ranking_2026 = pd.read_csv("./../data/2026_apr_ranking.csv")

In [41]:
wc_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,...,home_num_goals_scored_last_5_games,away_num_goals_scored_last_5_games,home_num_goals_conceded_last_5_games,away_num_goals_conceded_last_5_games,normalized_goals_for_home_team,normalized_goals_for_away_team,normalized_goals_against_home_team,normalized_goals_against_away_team,gf_pg_diff_last5,ga_pg_diff_last5
49215,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,1869.966473,...,7.0,6.0,1.0,8.0,1.4,1.2,0.2,1.6,0.2,-1.4
49216,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,1856.988740,...,5.0,12.0,5.0,6.0,1.0,2.4,1.0,1.2,-1.4,-0.2
49217,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,1807.261284,...,4.0,10.0,2.0,5.0,0.8,2.0,0.4,1.0,-1.2,-0.6
49218,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,1802.740129,...,11.0,5.0,10.0,7.0,2.2,1.0,2.0,1.4,1.2,0.6
49219,2026-06-13,Australia,Turkey,NaN,NaN,FIFA World Cup,Vancouver,Canada,True,1868.872547,...,7.0,10.0,7.0,3.0,1.4,2.0,1.4,0.6,-0.6,0.8


In [42]:
# The name of the countries in the ranking dataset and the matches dataset might not exactly match. 
# We will convert any of the mismatching country names among the countries that has already qualified for the world cup. 

df_ranking_2026.loc[df_ranking_2026["team"] == "Korea Republic", "team"] = "South Korea"
df_ranking_2026.loc[df_ranking_2026["team"] == "USA", "team"] = "United States"
df_ranking_2026.loc[df_ranking_2026["team"] == "IR Iran", "team"] = "Iran"
df_ranking_2026.loc[df_ranking_2026["team"] == "Cabo Verde", "team"] = "Cape Verde"
df_ranking_2026.loc[df_ranking_2026["team"] == "Côte d'Ivoire", "team"] = "Ivory Coast"
df_ranking_2026.loc[df_ranking_2026["team"] == "Curacao", "team"] = "Curaçao"
df_ranking_2026.loc[df_ranking_2026["team"] == "Czechia", "team"] = "Czech Republic"
df_ranking_2026.loc[df_ranking_2026["team"] == "Türkiye", "team"] = "Turkey"
df_ranking_2026.loc[df_ranking_2026["team"] == "Congo DR", "team"] = "DR Congo"

In [ ]:
# get the home and away team ranking. 
rank_lookup = df_ranking_2026.set_index("team")["rank"].to_dict()

wc_matches["home_rank"] = wc_matches["home_team"].map(rank_lookup)
wc_matches["away_rank"] = wc_matches["away_team"].map(rank_lookup)

/var/folders/sn/421wkqvd0ys33b2hmc60sldw0000gn/T/ipykernel_98015/372802791.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wc_matches["home_rank"] = wc_matches["home_team"].map(rank_lookup)
/var/folders/sn/421wkqvd0ys33b2hmc60sldw0000gn/T/ipykernel_98015/372802791.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wc_matches["away_rank"] = wc_matches["away_team"].map(rank_lookup)


In [ ]:
# generate the features just like how we did in the milestone notebook. 

wc_matches = wc_matches.copy()
wc_matches["is_friendly"] = wc_matches["tournament"] == "Friendly"
wc_matches["rank_diff"] = wc_matches["home_rank"] - wc_matches["away_rank"]
wc_matches["rank_sum"] = wc_matches["home_rank"] + wc_matches["away_rank"]
wc_matches["elo_sum"] = wc_matches["home_elo"] + wc_matches["away_elo"]

In [ ]:
# Get the 15 features list that we will be using for our model. 
final_features = ["home_team", "away_team", 
    "elo_diff", "elo_sum", "home_team_h2h_wins", "away_team_h2h_wins", "h2h_draws",
    "rank_diff", "rank_sum", "is_friendly", "home_count_last5", "away_count_last5",
    "pts_pg_diff_last5", "pts_pg_sum_last5", "gf_pg_diff_last5", "ga_pg_diff_last5", "neutral"
] 


In [ ]:
# Save the results to a csv file. We'll use this in the wc_2026_prediction notebook. 
wc_matches[final_features].to_csv("./../data/wc_matches_with_features.csv", index=False)

In [47]:
df_matches

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,...,home_num_goals_scored_last_5_games,away_num_goals_scored_last_5_games,home_num_goals_conceded_last_5_games,away_num_goals_conceded_last_5_games,normalized_goals_for_home_team,normalized_goals_for_away_team,normalized_goals_against_home_team,normalized_goals_against_away_team,gf_pg_diff_last5,ga_pg_diff_last5
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1500.840390,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1495.940229,...,2.0,4.0,4.0,2.0,1.000000,2.000000,2.000000,1.000000,-1.000000,1.000000
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1501.835139,...,5.0,4.0,4.0,5.0,1.666667,1.333333,1.333333,1.666667,0.333333,-0.333333
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1499.034368,...,6.0,7.0,7.0,6.0,1.500000,1.750000,1.750000,1.500000,-0.250000,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49282,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,1798.850043,...,3.0,3.0,3.0,2.0,0.600000,0.600000,0.600000,0.400000,0.000000,0.200000
49283,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,1750.245099,...,6.0,9.0,7.0,1.0,1.200000,1.800000,1.400000,0.200000,-0.600000,1.200000
49284,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True,1813.896239,...,7.0,7.0,2.0,2.0,1.400000,1.400000,0.400000,0.400000,0.000000,0.000000
49285,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,1913.245761,...,5.0,11.0,5.0,1.0,1.000000,2.200000,1.000000,0.200000,-1.200000,0.800000
